In [2]:
import rasterio as rio
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from download_utils import *
import requests
from pathlib import Path
from rasterio.mask import mask
from PIL import Image
import numpy as np

# Paths and parameters

In [ ]:
roi_path = 'data/vector_data/roi.gpkg' # path to the ROI bbox (.gpkg)
lidar_dir = 'data/lidar' # path to the output LiDAR images
image_dir = 'data/images' # path to the polygon LiDAR data (1 image / polygon)
polygons_path = 'data/vector_data/polygons.gpkg' # path to the polygons (.gpkg)

product='MNH' # product type : MNT / MNS / MNH
index_col='REFGDONSG' # name of the index in the polygons file 

# Get LiDAR acquisition information

In [ ]:
roi = gpd.read_file(roi_path)
minx, miny, maxx, maxy = roi.total_bounds

url = (
    "https://data.geopf.fr/wfs/ows?"
    "service=WFS&version=2.0.0&request=GetFeature"
    "&typeName=IGNF_NUAGES-DE-POINTS-LIDAR-HD:dalle"
    "&outputFormat=application/json"
    f"&bbox={minx},{miny},{maxx},{maxy},{roi.crs.to_string()}"
)

wfs_gdf = gpd.read_file(url)
wfs_gdf = wfs_gdf.to_crs(roi.crs)
df = gpd.sjoin(wfs_gdf, roi, predicate="intersects")

## Visualize acquisition dates

In [ ]:
df = unroll_metadata(df)
plot_acquisition_dates(df)

In [ ]:
year_tiles = get_year_data(df)
year_data = year_tiles[2023] # choose the wanted year 

# Download
1. Get download links for wanted product (MNH / MNT / MNS)
2. Download tiles

In [ ]:
year_data = get_product_urls(year_data, product, roi)
year_data = download_tiles(year_data, product, lidar_dir)

# Get image for each polygon

In [ ]:
get_polygon_images(polygons_path, lidar_dir, image_dir, year_data, index_col)